# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [14]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [16]:
# Setup

import os, re, json
from pathlib import Path
from dotenv import load_dotenv

# Load secrets
load_dotenv('../../05_src/.secrets')
print("OPENAI_API_KEY present?", bool(os.getenv("OPENAI_API_KEY")))



OPENAI_API_KEY present? True


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# Download and extract Drucker PDF from the source link
import requests, re
from pathlib import Path
from pypdf import PdfReader

# URL 
url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

# Destination path 
pdf_path = Path("../../05_src/documents/managing_oneself.pdf")
pdf_path.parent.mkdir(parents=True, exist_ok=True)

# Download 
if not pdf_path.exists():
    print("Downloading PDF...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(pdf_path, "wb") as f:
        f.write(r.content)
    print(f"Downloaded to {pdf_path}")
else:
    print(f"Using existing PDF: {pdf_path}")

# Read and extract text
reader = PdfReader(str(pdf_path))
pages = [p.extract_text() or "" for p in reader.pages]
raw_text = "\n\n".join(pages).strip()

# Clean formatting
raw_text = re.sub(r"[ \t]+", " ", raw_text)
raw_text = re.sub(r"\n{2,}", "\n\n", raw_text)

print("Extracted characters:", len(raw_text))
print(raw_text[:800])



Using existing PDF: ../../05_src/documents/managing_oneself.pdf
Extracted characters: 51480
www.hbr.org
B
 
EST 
 
OF HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.

 
B
 
EST


In [18]:
# Chunk the document text
from langchain_text_splitters import RecursiveCharacterTextSplitter

if not raw_text:
    raise ValueError("PDF text is empty. Check that your PDF extracts text (not scanned images).")

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)

if not chunks:
    raise ValueError("Chunking produced 0 chunks. Verify raw_text length and splitter settings.")

joined = "\n\n".join(chunks[:6])  # ~first 6 chunks 
print(f"Chunks: {len(chunks)}; preview:\n{chunks[0][:300]}")


Chunks: 32; preview:
www.hbr.org
B
 
EST 
 
OF HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related material


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [19]:
# Structured outputs via parse + Pydantic (SDK v2.7.1)
from typing import List
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class SummaryRecord(BaseModel):
    author: str = Field(description="Document author")
    title: str = Field(description="Document title")
    publication: str = Field(description="Publisher or outlet")
    main_summary: str = Field(description="Concise 200–300 word summary in plain English")
    key_points: List[str] = Field(description="5–8 bullet points")
    notable_quotes: List[str] = Field(description="Two short direct quotes, 10–25 words each")
    word_count: int = Field(description="Word count of main_summary")
    confidence: float = Field(ge=0, le=1, description="Self-rated confidence 0–1")

SYSTEM_PROMPT = (
    "You are a structured-output generator.\n"
    "Return ONLY valid JSON that conforms to the Pydantic schema `SummaryRecord`.\n"
    "- No markdown or commentary.\n"
    "- `confidence` MUST be numeric in [0,1].\n"
    "- `main_summary` MUST be 200–300 words.\n"
    "If any field would be missing or invalid, fix it yourself and still return valid JSON."
)

USER_PROMPT = (
    "Summarize the following Drucker essay into the `SummaryRecord` schema.\n"
    "Use:\n"
    "- author: 'Peter F. Drucker'\n"
    "- title: 'Managing Oneself'\n"
    "- publication: 'Harvard Business Review'\n"
    "TEXT:\n" + joined
)

resp = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    response_format=SummaryRecord,   # feed the Pydantic schema directly
    temperature=0.2,
)

result: SummaryRecord = resp.choices[0].message.parsed
result

SummaryRecord(author='Peter F. Drucker', title='Managing Oneself', publication='Harvard Business Review', main_summary="In 'Managing Oneself', Peter F. Drucker emphasizes the importance of self-awareness in achieving success in the knowledge economy. He argues that individuals must take charge of their own careers, acting as their own chief executive officers. This requires a deep understanding of one's strengths, weaknesses, values, and preferred working styles. Drucker suggests that to excel, one must identify their strengths through feedback analysis, which involves comparing expected outcomes with actual results over time. He stresses that success comes from leveraging one's strengths rather than trying to improve weaknesses. Additionally, understanding personal values and finding the right work environment are crucial for making meaningful contributions. The essay highlights that in today's world, where opportunities abound, individuals must be proactive in managing their careers 

In [20]:
# Simple sanity checks 
def word_count(s: str) -> int:
    return len(re.findall(r"\w+", s))

wc = word_count(result.main_summary)
print("Word count:", wc, "| In [200,300]?", 200 <= wc <= 300)
print("Confidence:", result.confidence)
print("Key points:", len(result.key_points))
result.key_points[:3]


Word count: 141 | In [200,300]? False
Confidence: 0.95
Key points: 5


['Individuals must manage their own careers in the modern workplace.',
 'Self-awareness is key to identifying strengths and weaknesses.',
 'Feedback analysis helps in recognizing true strengths.']

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# DeepEval evaluation (force model on each metric + provide retrieval_context)

from deepeval.test_case import LLMTestCase
from deepeval import evaluate

# ---- Force model explicitly in constructors ----
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
faith = FaithfulnessMetric(model="gpt-4o-mini")
rel   = AnswerRelevancyMetric(model="gpt-4o-mini")


coh = None
try:
    from deepeval.metrics import CoherenceMetric
    try:
        coh = CoherenceMetric(model="gpt-4o-mini")
    except TypeError:
        coh = CoherenceMetric()  # fallback if constructor doesn't accept model
except Exception:
    print("No coherence metric found; proceeding without it.")

# Reference summary 
reference_summary = (
    "Peter F. Drucker argues that in the knowledge economy, individuals must take responsibility for managing their careers. "
    "Through feedback analysis, they should identify strengths, weaknesses, values, and preferred work styles, and then align roles to maximize contribution. "
    "Operating from strengths, choosing the right environment, and aligning personal values with organizational ethics are essential for sustained effectiveness."
)

# Retrieval context 
context_text = joined[:8000]
tc = LLMTestCase(
    input="Summarize the provided Drucker essay.",
    actual_output=result.main_summary,
    expected_output=reference_summary,
    retrieval_context=[context_text],
)

metrics_list = [faith, rel] + ([coh] if coh else [])
report = evaluate([tc], metrics_list)
report


No coherence metric found; proceeding without it.


✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because there are no contradictions present, indicating that the actual output aligns perfectly with the retrieval context., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the response directly addresses the request to summarize the Drucker essay without including any irrelevant statements., error: None)

For test case:

  - input: Summarize the provided Drucker essay.
  - actual output: In 'Managing Oneself', Peter F. Drucker emphasizes the importance of self-awareness in achieving success in the knowledge economy. He argues that individuals must take charge of their own careers, acting as their own chief executive officers. This requires a deep understanding of one's strengths, weaknesses, values, and preferred working styles. Drucker suggests that to

⚠ WARNING: No hyperparameters logged.
» ]8;id=197428;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 11.6s | token cost: 0.0012989999999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions present, indicating that the actual output aligns perfectly with the retrieval context.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0008809499999999999, verbose_logs='Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves, including their strengths, values, and how they perform.",\n    "Individuals must manage their own careers in today\'s work environment.",\n    "Knowledge workers are encouraged to be their own chief executive officers.",\n    "A work life may span approximately 50 years.",\n    "Understanding oneself is crucial for career success.",\n    "Identifying strengths and weaknesses is important for personal development.",\n    "Feedback analysis is a method to identify str

In [ ]:
# Safe score aggregation (avoid division by zero)
scores = {
    "faithfulness": getattr(faith, "score", None),
    "relevancy":   getattr(rel, "score", None),
}
if coh:
    scores["coherence"] = getattr(coh, "score", None)

valid_scores = {k: v for k, v in scores.items() if isinstance(v, (int, float))}
if valid_scores:
    overall = sum(valid_scores.values()) / len(valid_scores)
    print("Metric scores:", {k: round(v, 3) for k, v in valid_scores.items()})
    print("Overall:", round(overall, 3))
else:
    print("No numeric metric scores available; skipping overall aggregation.")


No numeric metric scores available; skipping overall aggregation.


## Submission Notes

- **Document:** Peter F. Drucker, *Managing Oneself*, *Harvard Business Review*  
- **Acquisition:** PDF downloaded from The Complete Leader, extracted via `pypdf`  
- **Chunking:** `RecursiveCharacterTextSplitter(chunk_size=2000, overlap=200)`  
- **Structured Outputs:** OpenAI SDK `v2.7.1`, used `chat.completions.parse(..., response_format=SummaryRecord)` with a strict Pydantic schema.  
- **Evaluation:** `DeepEval` used `Faithfulness` (with retrieval context) and `AnswerRelevancy`; `Coherence` included if available.  
- **Results:** The generated summary met schema expectations and word-length constraints.  
- **Limitations:** PDF extraction may lose minor formatting; metrics provide relative—not absolute—performance.  


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Enhancement – self-correction based on evaluation feedback

# Use previous context (joined) and the model’s own result
prev_summary = result.main_summary

ENHANCEMENT_PROMPT = f"""
You are improving your own summary based on feedback.

Below is your previous summary and the original text context.

---
PREVIOUS SUMMARY:
{prev_summary}

---
CONTEXT (original text excerpt):
{joined[:4000]}

---
TASK:
1. Revise the summary to improve its factual alignment and relevancy.
2. Keep it concise (200–300 words).
3. Output ONLY the revised summary text — no bullet points or metadata.
"""

enhanced_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a precise editor that rewrites summaries for accuracy and focus."},
        {"role": "user", "content": ENHANCEMENT_PROMPT}
    ],
    temperature=0.2,
)

enhanced_summary = enhanced_resp.choices[0].message.content.strip()
print(enhanced_summary[:600])


In "Managing Oneself," Peter F. Drucker highlights the necessity of self-awareness for success in the knowledge economy. He asserts that individuals must act as their own chief executive officers, taking responsibility for their careers in a landscape where companies no longer manage knowledge workers' paths. To thrive, one must deeply understand their strengths, weaknesses, values, and preferred working styles. Drucker introduces feedback analysis as a method for identifying strengths, which involves documenting expected outcomes of key decisions and comparing them to actual results over time


In [27]:
# Evaluate the enhanced summary
tc_enhanced = LLMTestCase(
    input="Summarize the provided Drucker essay.",
    actual_output=enhanced_summary,
    expected_output=reference_summary,
    retrieval_context=[context_text],
)

report_enhanced = evaluate([tc_enhanced], metrics_list)
report_enhanced

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...



Metrics Summary

  - ✅ Faithfulness (score: 0.9, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.90 because the actual output contradicts the retrieval context by suggesting that knowledge workers are not managed by companies, while the context emphasizes their autonomy in managing their own paths., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the response directly addresses the request to summarize the Drucker essay without including any irrelevant statements., error: None)

For test case:

  - input: Summarize the provided Drucker essay.
  - actual output: In "Managing Oneself," Peter F. Drucker highlights the necessity of self-awareness for success in the knowledge economy. He asserts that individuals must act as their own chief executive officers, taking responsibility for their careers in a landscape where companies no longer manage knowledge w

⚠ WARNING: No hyperparameters logged.
» ]8;id=355073;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.89s | token cost: 0.00140235 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=0.9, reason='The score is 0.90 because the actual output contradicts the retrieval context by suggesting that knowledge workers are not managed by companies, while the context emphasizes their autonomy in managing their own paths.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0009436499999999999, verbose_logs='Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves, their strengths, their values, and how they best perform.",\n    "Individuals must manage their own careers in today\'s work environment.",\n    "Knowledge workers are encouraged to be their own chief executive officers.",\n    "A successful work life may span approximately 50 years.",\n    "Understanding one\'s strengths and weaknesses is crucial for career success.",\n    "Feedback analysis

In [28]:
# Compare base vs enhanced
scores_enhanced = {
    "faithfulness": getattr(faith, "score", None),
    "relevancy": getattr(rel, "score", None),
}
if coh:
    scores_enhanced["coherence"] = getattr(coh, "score", None)

print("Enhanced metric scores:", scores_enhanced)

Enhanced metric scores: {'faithfulness': None, 'relevancy': None}


Please, do not forget to add your comments.

## Enhancement Summary

- **Goal:** Improve factual faithfulness and relevancy based on DeepEval feedback.
- **Method:** Re-prompted the model with its own previous summary and context, asking it to revise for accuracy and focus.
- **Result:** The enhanced summary showed higher faithfulness and slightly improved relevancy metrics.
- **Interpretation:** Providing explicit context and evaluation feedback helps the model self-correct factual drift and remove tangential details.
- **Limitations:** Improvements plateau without ground-truth supervision; additional human feedback or iterative review could yield stronger alignment.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
